## Failure Injection Cheat Sheet

Use the widgets at the top of this notebook to simulate realistic pipeline failures. Set `failure_mode` to `none` for a clean run.

| Failure Mode | Target | Error Signature | RCA Pattern |
| --- | --- | --- | --- |
| `timeout` | `placement_details` | `RuntimeError: Failed to fetch ... after 5 attempts` | API unresponsive, retry exhaustion |
| `server_500` | `placement_details` | Same RuntimeError (retries exhaust on 500s) | Upstream server failure |
| `connection_error` | `placement_details` | Same RuntimeError (network-level failure) | DNS/connectivity issue |
| `bad_xml` | `placements` | `RuntimeError: Failed to parse XML: mismatched tag` | Data corruption / API returning garbage |
| `auth_401` | `placements` | `HTTPError: 401 Client Error` | Token expiry mid-pipeline |

**Notes:**
* `bad_xml` and `auth_401` must target `placements` or `insertion_orders` (where XML parsing / raise_for_status occurs). Targeting `placement_details` with these modes won't crash because that level stores raw payloads without parsing.
* `failure_after_calls` controls how many successful calls happen before the failure fires (partial progress for richer logs).
* `failure_once=true` fails one logical API call (including all its retries) then lets the rest succeed. Set to `false` to fail ALL calls after the threshold (caution: may cause long timeouts).


In [ ]:
import sys
sys.path.insert(0, "/Workspace/Users/saurabh.burewar@wbd.com/foresite_digital/data_engineering")

import json
import logging
import requests
import time
import random
import pandas as pd
import xml.etree.ElementTree as ET
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import List, Dict, Optional, Tuple
from ingest_utils import (
    AUTH_URL, BASE_URL,
    PLACEMENTS_URL_TEMPLATE, PLACEMENT_DETAILS_TEMPLATE,
    MAX_RETRY_ATTEMPTS, TIMEOUT, MAX_WORKERS,
    get_secret,
    get_fw_access_token,
)


In [ ]:
incremental = True  # Set to False for a full load (no date filter)

# ---------------- API Config ----------------
CAMPAIGN_ID = "76354594"
EXCLUDE_IO_IDS = {"87148837"}
INSERTION_ORDERS_ENDPOINT = f"{BASE_URL}/campaign/{CAMPAIGN_ID}/insertion_orders"


In [ ]:
# ---------------- Logging Setup ----------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

# ---------------- Environment Setup ----------------
try:
    env = dbutils.widgets.get("environment")
except Exception:
    env = "dev"
    logging.warning(f"Environment not specified, defaulting to {env}")
logging.info(f"Running in environment: {env}")

# ---------------- AWS Secrets Manager ----------------
aws_secret_name = f"{env}/foresite_digital_databricks_data_engineering"
aws_secrets = get_secret(aws_secret_name)

# ---------------- API Authentication ----------------
token_info = get_fw_access_token(aws_secrets)
HEADERS = {"accept": "application/xml", "Authorization": f"Bearer {token_info.get('access_token')}"}

# ---------------- HTTP Session ----------------
http_session = requests.Session()
http_session.headers.update(HEADERS)
adapter = requests.adapters.HTTPAdapter(max_retries=3)
http_session.mount("https://", adapter)


2026-08-06 07:02:53,862 [INFO] Running in environment: dev
2026-08-06 07:02:53,876 [INFO] Found credentials from IAM Role: databricks-s3-role


In [ ]:
# ---------------- Failure Injection ----------------
# Purpose: inject realistic dependency failures without changing the pipeline logic.
# Default is disabled. When enabled, this monkey-patches the HTTP session used by the notebook.

def _ensure_dropdown(name: str, default_value: str, choices: list[str], label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default_value, choices, label)


def _ensure_text(name: str, default_value: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default_value, label)


_ensure_dropdown("failure_mode", "none", ["none", "timeout", "auth_401", "server_500", "bad_xml", "connection_error"], "Failure mode")
_ensure_text("failure_after_calls", "0", "Fail after N matching calls")
_ensure_dropdown("failure_target", "placement_details", ["all", "insertion_orders", "placements", "placement_details"], "Failure target")
_ensure_dropdown("failure_once", "true", ["true", "false"], "Fail once")

failure_mode = dbutils.widgets.get("failure_mode").strip().lower()
failure_target = dbutils.widgets.get("failure_target").strip().lower()
failure_once = dbutils.widgets.get("failure_once").strip().lower() == "true"
try:
    failure_after_calls = max(0, int(dbutils.widgets.get("failure_after_calls")))
except Exception:
    failure_after_calls = 0


def _target_matches(url: str) -> bool:
    if failure_target == "all":
        return True
    if failure_target == "insertion_orders":
        return "/insertion_orders" in url
    if failure_target == "placements":
        return "/placements" in url and "placement/" not in url
    if failure_target == "placement_details":
        return "/placement/" in url
    return False


def _make_response(url: str, status_code: int, body: str, content_type: str = "application/xml") -> requests.Response:
    resp = requests.Response()
    resp.status_code = status_code
    resp._content = body.encode("utf-8")
    resp.url = url
    resp.headers["Content-Type"] = content_type
    return resp


# Reset any prior patch so the cell can be re-run safely with different widget values.
if hasattr(http_session, "_original_request"):
    http_session.request = http_session._original_request
else:
    http_session._original_request = http_session.request

_failure_state = {
    "matching_calls": 0,
    "unique_failures": 0,
    "failed_urls": set(),
    "mode": failure_mode,
    "after_calls": failure_after_calls,
    "target": failure_target,
    "once": failure_once,
}


def _request_with_failure_injection(method, url, **kwargs):
    if _failure_state["mode"] == "none" or not _target_matches(url):
        return http_session._original_request(method, url, **kwargs)

    # If this URL was already failed, keep failing it (retries must also fail to exhaust retry budget)
    is_retry_of_failed = url in _failure_state["failed_urls"]

    if not is_retry_of_failed:
        _failure_state["matching_calls"] += 1
        matching_call_num = _failure_state["matching_calls"]

        if matching_call_num <= _failure_state["after_calls"]:
            return http_session._original_request(method, url, **kwargs)

        # In "once" mode, only fail one unique URL (but keep failing its retries)
        if _failure_state["once"] and _failure_state["unique_failures"] >= 1:
            return http_session._original_request(method, url, **kwargs)

        # Mark this URL as failed
        _failure_state["unique_failures"] += 1
        _failure_state["failed_urls"].add(url)

    logging.warning(
        "Injecting failure mode=%s target=%s url=%s retry=%s",
        _failure_state["mode"],
        _failure_state["target"],
        url,
        is_retry_of_failed,
    )

    if _failure_state["mode"] == "timeout":
        raise requests.Timeout(f"Injected timeout for {url}")
    if _failure_state["mode"] == "connection_error":
        raise requests.ConnectionError(f"Injected connection error for {url}")
    if _failure_state["mode"] == "auth_401":
        return _make_response(url, 401, "<error><message>Injected expired token</message></error>")
    if _failure_state["mode"] == "server_500":
        return _make_response(url, 500, "<error><message>Injected upstream server failure</message></error>")
    if _failure_state["mode"] == "bad_xml":
        return _make_response(url, 200, "<response><broken></response>")

    return http_session._original_request(method, url, **kwargs)


http_session.request = _request_with_failure_injection
logging.info(
    "Failure injection configured: mode=%s, target=%s, after_calls=%s, once=%s",
    failure_mode,
    failure_target,
    failure_after_calls,
    failure_once,
)


2026-08-06 07:02:54,255 [INFO] Received command c on object id p1
2026-08-06 07:02:54,270 [INFO] Failure injection configured: mode=timeout, target=placement_details, after_calls=0, once=True


In [ ]:
# ---------------- Data Models ----------------
@dataclass
class InsertionOrder:
    id: str
    name: str
    status: str
    created_at: str
    updated_at: str

# ---------------- Utility Functions ----------------

def _parse_xml(text: bytes) -> ET.Element:
    try:
        return ET.fromstring(text)
    except ET.ParseError as e:
        raise RuntimeError(f"Failed to parse XML: {e}")


def _safe_text(node: ET.Element) -> str:
    return node.text.strip() if node is not None and node.text else ""


def _extract_pagination(root: ET.Element, current_page: int) -> Tuple[int, int]:
    try:
        return int(root.attrib.get("current_page", current_page)), int(root.attrib.get("total_pages", current_page))
    except ValueError:
        return current_page, current_page

# ---------------- Retry Helper ----------------
def _retry_request(method: str, url: str, params: Optional[Dict] = None, data: Optional[Dict] = None, timeout: int = TIMEOUT, max_attempts: int = MAX_RETRY_ATTEMPTS, backoff_factor: float = 1.0, allowed_statuses: Tuple[int, ...] = (500, 502, 503, 504)) -> requests.Response:
    """
    Perform an HTTP request with simple exponential backoff on transient errors.
    - Retries on network exceptions and on server 5xx statuses listed in allowed_statuses.
    - Returns the Response for non-retriable statuses (e.g., 4xx) so callers can handle them.
    """
    attempt = 1
    while attempt <= max_attempts:
        try:
            resp = http_session.request(method, url, params=params, data=data, timeout=timeout)
            if resp.status_code < 400 or resp.status_code not in allowed_statuses:
                return resp
            logging.warning(f"Transient status {resp.status_code} for {url} attempt {attempt}/{max_attempts}")
        except requests.RequestException as exc:
            logging.warning(f"Request exception for {url}: {exc} (attempt {attempt}/{max_attempts})")
        sleep_time = backoff_factor * (2 ** (attempt - 1))
        jitter = random.uniform(0, sleep_time * 0.1)
        time.sleep(sleep_time + jitter)
        attempt += 1
    raise RuntimeError(f"Failed to fetch {url} after {max_attempts} attempts")


def _fetch_paginated(endpoint: str, tag: str, fields: List[str], per_page: int = 50) -> List[Dict[str, str]]:
    results, page = [], 1
    while True:
        resp = _retry_request("GET", endpoint, params={"page": page, "per_page": per_page}, timeout=TIMEOUT)
        resp.raise_for_status()
        root = _parse_xml(resp.content)
        for item in root.findall(f".//{tag}"):
            record = {field: _safe_text(item.find(field)) for field in fields}
            if record.get("id"):
                results.append(record)
        current_page, total_pages = _extract_pagination(root, page)
        if current_page >= total_pages:
            break
        page += 1
    return results


2026-08-06 07:02:54,360 [INFO] Received command c on object id p1


In [ ]:
def should_fetch_placement(placement_id: str, listing_updated_at: Optional[str], current_map: Dict[str, Optional[str]]) -> bool:
    """Return True if we should fetch detailed placement data.

    Skip fetch when we already have a stored PLACEMENT_UPDATED_AT that is >= the listing's updated_at.
    Uses pandas.to_datetime(..., utc=True, errors='coerce') for robust parsing.
    """
    cur_val = current_map.get(str(placement_id))
    if cur_val is None:
        return True
    try:
        cur_dt = pd.to_datetime(cur_val, utc=True, errors='coerce')
    except Exception:
        cur_dt = pd.NaT
    try:
        listing_dt = pd.to_datetime(listing_updated_at, utc=True, errors='coerce')
    except Exception:
        listing_dt = pd.NaT
    if pd.isna(listing_dt):
        logging.info(f"Listing updated_at missing/unparseable for placement {placement_id}; fetching details")
        return True
    if pd.isna(cur_dt):
        logging.info(f"Stored updated_at unparsable for placement {placement_id}; fetching details")
        return True
    return listing_dt > cur_dt


def get_all_placements_for_order(order_id: str, query_string: str) -> List[Dict[str, str]]:
    endpoint = PLACEMENTS_URL_TEMPLATE.format(id=order_id, query_param_list=query_string)
    return _fetch_paginated(endpoint, "placement", ["id", "name", "status", "created_at", "updated_at"])


def get_placement_details(placement_id: str, insertion_order_id: str, created_at: Optional[str] = None,
        updated_at: Optional[str] = None) -> Dict[str, str]:
    """Fetch full placement details and attach listing-created/updated timestamps when available."""
    endpoint = PLACEMENT_DETAILS_TEMPLATE.format(placement_id=placement_id)
    resp = _retry_request("GET", endpoint, timeout=TIMEOUT)
    resp.raise_for_status()
    return {
        "PAYLOAD": resp.content.decode("utf-8"),
        "INGESTION_TS": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S.%f"),
        "INSERTION_ORDER_ID": insertion_order_id,
        "CAMPAIGN_ID": CAMPAIGN_ID,
        "PLACEMENT_ID": placement_id,
        "PLACEMENT_CREATED_AT": created_at,
        "PLACEMENT_UPDATED_AT": updated_at,
        "PROCESS_STATUS": "INGESTED"
    }


2026-08-06 07:02:54,456 [INFO] Received command c on object id p1


In [ ]:
raw_orders = _fetch_paginated(INSERTION_ORDERS_ENDPOINT, "insertion_order", ["id", "name", "status", "created_at", "updated_at"])
filtered_orders = [r for r in raw_orders if r.get("id") and str(r.get("id")) not in EXCLUDE_IO_IDS]
if len(filtered_orders) != len(raw_orders):
    logging.info("Excluded insertion order 87148837 from results")
orders = [InsertionOrder(**order) for order in filtered_orders]
logging.info(f"Fetched {len(orders)} insertion orders:\n" + "\n".join(f"  {o.id} - {o.name} ({o.status})" for o in orders))


2026-08-06 07:02:54,556 [INFO] Received command c on object id p1
2026-08-06 07:02:54,803 [INFO] Received command c on object id p0
2026-08-06 07:02:54,813 [INFO] Excluded insertion order 87148837 from results
2026-08-06 07:02:54,813 [INFO] Fetched 6 insertion orders:
  93327466 - FORECASTING TESTS (IN_ACTIVE)
  95922596 - ENT P&I Test Campaign - 5 (IN_ACTIVE)
  94047280 - ENT P&I Test Campaign - 4 (IN_ACTIVE)
  92336848 - ENT P&I Test Campaign - 3 (IN_ACTIVE)
  88951896 - ENT P&I Test Campaign - 2 (IN_ACTIVE)
  87268040 - ENT P&I Test Campaign - 1 (IN_ACTIVE)


In [ ]:
incremental = locals().get("incremental", False)  # Defaults to full load if not set

if not incremental:
    query_string = ""  # Full load: no filters applied
else:
    current_time = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
    query_string = f"end_date={current_time}.."
    logging.info(f"query_string: {query_string}")


2026-08-06 07:02:54,856 [INFO] Received command c on object id p1
/root/.ipykernel/376720/command-6804725387757166-1449257977:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  current_time = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")
2026-08-06 07:02:54,861 [INFO] query_string: end_date=2026-08-06T07:02:54Z..


In [ ]:
# Full load mode: no existing placement data to compare against
# (Snowflake lookup removed — all placements will be fetched)
current_map: Dict[str, Optional[str]] = {}
logging.info("Running in full-load mode. All placements will be fetched.")


2026-08-06 07:02:54,956 [INFO] Received command c on object id p1
2026-08-06 07:02:54,961 [INFO] Running in full-load mode. All placements will be fetched.


In [ ]:
all_placements = []
failed_placements = []

for io in orders:
    try:
        placements_raw = get_all_placements_for_order(io.id, query_string)
        logging.info(len(placements_raw))

        placements_to_fetch = []
        for p in placements_raw:
            pid = p.get("id")
            listing_updated = p.get("updated_at")
            if should_fetch_placement(pid, listing_updated, current_map):
                placements_to_fetch.append(p)
            else:
                logging.info(f"Skipping placement {pid}: stored updated_at >= listing updated_at")

        if not placements_to_fetch:
            logging.info(f"No placements to fetch for IO {io.id}")
            continue

        logging.info(f"Fetching {len(placements_to_fetch)} placements for IO {io.id}")
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(get_placement_details, p["id"], io.id, p.get("created_at"), p.get("updated_at")): p["id"] for p in placements_to_fetch}
            for future in as_completed(futures):
                placement_id = futures[future]
                try:
                    all_placements.append(future.result())
                except Exception as e:
                    failed_placements.append({"placement_id": placement_id, "io_id": io.id, "error": str(e)})
                    logging.error(f"Failed to fetch placement {placement_id} for IO {io.id}: {e}")
    except requests.HTTPError as e:
        logging.error(f"Failed to fetch placements for IO {io.id}: {e}")

logging.info(f"Total placements fetched: {len(all_placements)}, failed: {len(failed_placements)}")

if failed_placements:
    for fp in failed_placements:
        logging.error(f"  FAILED: placement_id={fp['placement_id']} io_id={fp['io_id']} error={fp['error']}")
    raise RuntimeError(
        f"Pipeline failed: {len(failed_placements)} placement(s) could not be fetched after retries. "
        f"First failure: {failed_placements[0]['error']}"
    )


2026-08-06 07:02:55,058 [INFO] Received command c on object id p1
2026-08-06 07:02:55,193 [INFO] 13
2026-08-06 07:02:55,194 [INFO] Fetching 13 placements for IO 93327466
2026-08-06 07:02:55,194 [WARNING] Injecting failure mode=timeout target=placement_details url=https://api.freewheel.tv/services/v3/placement/95875283?show=all retry=False
2026-08-06 07:02:55,196 [WARNING] Request exception for https://api.freewheel.tv/services/v3/placement/95875283?show=all: Injected timeout for https://api.freewheel.tv/services/v3/placement/95875283?show=all (attempt 1/5)
2026-08-06 07:02:55,803 [INFO] Received command c on object id p0
2026-08-06 07:02:56,269 [WARNING] Injecting failure mode=timeout target=placement_details url=https://api.freewheel.tv/services/v3/placement/95875283?show=all retry=True
2026-08-06 07:02:56,269 [WARNING] Request exception for https://api.freewheel.tv/services/v3/placement/95875283?show=all: Injected timeout for https://api.freewheel.tv/services/v3/placement/95875283?sh

RuntimeError: Pipeline failed: 1 placement(s) could not be fetched after retries. First failure: Failed to fetch https://api.freewheel.tv/services/v3/placement/95875283?show=all after 5 attempts

In [ ]:
if all_placements:
    df = pd.DataFrame(all_placements)
    display(df)
    logging.info(f"{len(all_placements)} placements loaded successfully (read-only, no writes performed)")
else:
    logging.warning("No placements fetched.")


Command skipped
